# Voyage AI — Exploratory Data Analysis & Model Training

This notebook covers:
1. Loading and inspecting the tourist places dataset
2. EDA — distributions, correlations, category analysis
3. Feature Engineering
4. Model Training (TF-IDF + Cosine Similarity)
5. Model Evaluation with sample queries
6. Saving the trained model


In [ ]:
import sys
sys.path.insert(0, '../..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import json

plt.style.use('seaborn-v0_8')
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', 80)
print('Libraries loaded ✅')

## 1. Load Dataset

In [ ]:
df = pd.read_csv('../datasets/tourist_places.csv')
print(f'Shape: {df.shape}')
df.head()

In [ ]:
print('\nData Types:')
print(df.dtypes)
print('\nMissing Values:')
print(df.isnull().sum())
print('\nBasic Statistics:')
df.describe()

## 2. EDA — Distributions & Insights

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Rating distribution
axes[0,0].hist(df['rating'].dropna(), bins=15, color='#4F46E5', edgecolor='white')
axes[0,0].set_title('Rating Distribution', fontsize=13)
axes[0,0].set_xlabel('Rating')

# Category distribution
df['category'].value_counts().plot(kind='bar', ax=axes[0,1], color='#10B981')
axes[0,1].set_title('Places by Category', fontsize=13)
axes[0,1].tick_params(axis='x', rotation=30)

# City distribution
df['city'].value_counts().head(10).plot(kind='barh', ax=axes[1,0], color='#F59E0B')
axes[1,0].set_title('Top 10 Cities by Place Count', fontsize=13)

# Entry fee distribution
df['entry_fee'].fillna(0).plot(kind='hist', ax=axes[1,1], bins=15, color='#EF4444')
axes[1,1].set_title('Entry Fee Distribution', fontsize=13)

plt.tight_layout()
plt.savefig('../notebooks/eda_plots.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Indoor vs Outdoor
indoor_counts = df['is_indoor'].value_counts()
plt.figure(figsize=(6, 4))
plt.pie(indoor_counts, labels=['Outdoor', 'Indoor'], autopct='%1.1f%%',
        colors=['#10B981', '#4F46E5'])
plt.title('Indoor vs Outdoor Attractions')
plt.show()

print(f'\nAverage rating: {df["rating"].mean():.2f}')
print(f'Average duration: {df["estimated_duration_hours"].mean():.1f} hours')
print(f'Cities covered: {df["city"].nunique()}')
print(f'Categories: {df["category"].unique()}')

## 3. Feature Engineering

In [ ]:
def parse_tags(val):
    if isinstance(val, list): return val
    try: return json.loads(str(val).replace("'", '"'))
    except: return []

df['tags'] = df['tags'].apply(parse_tags)
df['description'] = df['description'].fillna('')
df['category'] = df['category'].fillna('general')

def build_feature_string(row):
    parts = []
    parts.extend([str(row['category'])] * 3)  # category has 3x weight
    if isinstance(row['tags'], list):
        parts.extend(row['tags'] * 2)
    parts.append(str(row.get('city', '')))
    parts.append(str(row.get('state', '')))
    parts.append(str(row['description']))
    return ' '.join(parts).lower()

df['features'] = df.apply(build_feature_string, axis=1)
print('Sample feature string:')
print(df['features'].iloc[0])

## 4. Model Training — TF-IDF + Cosine Similarity

In [ ]:
vectorizer = TfidfVectorizer(ngram_range=(1, 2), min_df=1, max_df=0.95, stop_words='english')
tfidf_matrix = vectorizer.fit_transform(df['features'])

print(f'TF-IDF Matrix shape: {tfidf_matrix.shape}')
print(f'Vocabulary size: {len(vectorizer.vocabulary_)}')

# Top features
feature_names = vectorizer.get_feature_names_out()
print(f'Sample features: {feature_names[:20]}')

## 5. Evaluation — Sample Queries

In [ ]:
def recommend(interests, city=None, top_n=5, weather_bad=False):
    query = ' '.join(interests).lower()
    query_vec = vectorizer.transform([query])
    
    filter_df = df.copy()
    if city:
        filter_df = filter_df[filter_df['city'].str.lower().str.contains(city.lower(), na=False)]
    if weather_bad:
        indoor_mask = filter_df['is_indoor'] == True
        if indoor_mask.sum() >= 2:
            filter_df = filter_df[indoor_mask]
    
    if filter_df.empty:
        return pd.DataFrame()
    
    idx = filter_df.index.tolist()
    sims = cosine_similarity(query_vec, tfidf_matrix[idx]).flatten()
    filter_df = filter_df.copy()
    filter_df['score'] = sims
    return filter_df.sort_values(['score', 'rating'], ascending=False).head(top_n)[['name', 'city', 'category', 'rating', 'score']]

test_cases = [
    (['beach', 'water_sports', 'adventure'], 'Goa'),
    (['history', 'heritage', 'monument'], 'Agra'),
    (['nature', 'wildlife'], None),
    (['temple', 'spiritual'], None),
    (['museum', 'indoor'], 'Delhi'),
]

for interests, city in test_cases:
    print(f'\n🔍 Query: {interests}, city={city}')
    result = recommend(interests, city)
    print(result.to_string(index=False))

In [ ]:
# Similarity heatmap for first 10 places
sim_matrix = cosine_similarity(tfidf_matrix[:10], tfidf_matrix[:10])
plt.figure(figsize=(10, 8))
sns.heatmap(
    sim_matrix,
    xticklabels=df['name'].iloc[:10].str[:20],
    yticklabels=df['name'].iloc[:10].str[:20],
    annot=True, fmt='.2f', cmap='RdYlGn'
)
plt.title('Content Similarity Heatmap (First 10 Places)')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.savefig('../notebooks/similarity_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. Save Trained Model

In [ ]:
import joblib
from pathlib import Path

Path('../models').mkdir(exist_ok=True)

model_data = {
    'vectorizer': vectorizer,
    'tfidf_matrix': tfidf_matrix,
    'places_df': df,
}

joblib.dump(model_data, '../models/recommendation_model.pkl')
print('✅ Model saved to ../models/recommendation_model.pkl')
print(f'Model file size: {Path("../models/recommendation_model.pkl").stat().st_size / 1024:.1f} KB')